# **Bioinformatics Project - Computational Drug Discovery [Part 3] Descriptor Calculation and Dataset Preparation**


## **Download PaDEL-Descriptor**

In [3]:
!pip install padelpy

In [15]:
from padelpy import padeldescriptor

padeldescriptor(
    mol_dir='molecule.smi',
    d_file='descriptors_output.csv',
    fingerprints=True,
    removesalt=True
)

## **Load bioactivity data**

Download the curated ChEMBL bioactivity data that has been pre-processed from Parts 1 and 2 of this Bioinformatics Project series. Here we will be using the **bioactivity_data_3class_pIC50.csv** file that essentially contain the pIC50 values that we will be using for building a regression model.

In [6]:
import pandas as pd

In [7]:
df3 = pd.read_csv('bioactivity_data_2class.csv')

In [8]:
df3

,molecule_chembl_id,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class
0,CHEMBL2402206,Cc1ccc(S(=O)(=O)c2cnc(SCC(=O)Nc3ccccc3C(F)(F)F...,497.520,3.96914,2.0,6.0,3.928118,inactive
1,CHEMBL2402205,CC(C)C[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](...,1858.036,-4.81120,24.0,24.0,7.481486,active
2,CHEMBL3237245,COc1ccc(S(=O)(=O)N(CC(=O)O)c2ccc(N(CC(=O)O)S(=...,614.654,3.41680,2.0,8.0,7.698970,active
3,CHEMBL3632707,CCOC(=O)CN(c1ccc(N(CC(=O)OCC)S(=O)(=O)c2ccc(OC...,670.762,4.37380,0.0,10.0,3.602060,inactive
4,CHEMBL3632711,COc1ccc(S(=O)(=O)N(CC(N)=O)c2ccc(N(CC(N)=O)S(=...,612.686,2.21820,2.0,8.0,7.356547,active
...,...,...,...,...,...,...,...,...
259,CHEMBL3237245,COc1ccc(S(=O)(=O)N(CC(=O)O)c2ccc(N(CC(=O)O)S(=...,614.654,3.41680,2.0,8.0,7.619789,active
260,CHEMBL3237245,COc1ccc(S(=O)(=O)N(CC(=O)O)c2ccc(N(CC(=O)O)S(=...,614.654,3.41680,2.0,8.0,8.113509,active
261,CHEMBL5272223,CC(=O)N[C@@H](CC(=O)O)C(=O)N1CSC[C@H]1C(=O)N[C...,859.909,-2.91190,11.0,13.0,7.508638,active
262,CHEMBL1232461,CCNC(=O)C[C@@H]1N=C(c2ccc(Cl)cc2)c2cc(OC)ccc2-...,423.904,3.65602,1.0,5.0,5.000000,inactive


In [9]:
selection = ['canonical_smiles','molecule_chembl_id']
df3_selection = df3[selection]
df3_selection.to_csv('molecule.smi', sep='\t', index=False, header=False)

In [10]:
! cat molecule.smi | head -5

Cc1ccc(S(=O)(=O)c2cnc(SCC(=O)Nc3ccccc3C(F)(F)F)[nH]c2=O)c(C)c1	CHEMBL2402206
CC(C)C[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](CCC(=O)O)NC(=O)CNC(=O)[C@@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](CC(=O)O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC(N)=O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC(N)=O)NC(=O)[C@H](C)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](C)N)[C@@H](C)O)C(=O)O	CHEMBL2402205
COc1ccc(S(=O)(=O)N(CC(=O)O)c2ccc(N(CC(=O)O)S(=O)(=O)c3ccc(OC)cc3)c3ccccc23)cc1	CHEMBL3237245
CCOC(=O)CN(c1ccc(N(CC(=O)OCC)S(=O)(=O)c2ccc(OC)cc2)c2ccccc12)S(=O)(=O)c1ccc(OC)cc1	CHEMBL3632707
COc1ccc(S(=O)(=O)N(CC(N)=O)c2ccc(N(CC(N)=O)S(=O)(=O)c3ccc(OC)cc3)c3ccccc23)cc1	CHEMBL3632711


In [11]:
! cat molecule.smi | wc -l

264


## **Calculate fingerprint descriptors**


In [12]:
%%writefile padel.sh
java -Xms512m -Xmx512m -jar PaDEL-Descriptor/PaDEL-Descriptor.jar -removesalt -retained3d-descriptors -file descriptors_output.csv -dir ./ -fingerprints

Writing padel.sh


### **Calculate PaDEL descriptors**

In [13]:
! cat padel.sh

java -Xms512m -Xmx512m -jar PaDEL-Descriptor/PaDEL-Descriptor.jar -removesalt -retained3d-descriptors -file descriptors_output.csv -dir ./ -fingerprints


In [16]:
! ls -l

total 572
-rw-r--r-- 1 root root  53072 Sep  7 17:13 bioactivity_data_2class.csv
-rw-r--r-- 1 root root 480740 Sep  7 17:16 descriptors_output.csv
-rw-r--r-- 1 root root  34808 Sep  7 17:13 molecule.smi
-rw-r--r-- 1 root root    153 Sep  7 17:13 padel.sh
drwxr-xr-x 1 root root   4096 Aug 24 13:21 sample_data


## **Preparing the X and Y Data Matrices**

### **X data matrix**

In [17]:
df3_X = pd.read_csv('descriptors_output.csv')

In [18]:
df3_X

,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL2402206,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL2402205,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL3237245,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL3632707,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL3632711,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,CHEMBL3237245,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
260,CHEMBL3237245,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
261,CHEMBL5272223,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
262,CHEMBL1232461,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
df3_X = df3_X.drop(columns=['Name'])
df3_X

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
260,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
261,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
262,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


## **Y variable**

### **Convert IC50 to pIC50**

In [20]:
df3_Y = df3['pIC50']
df3_Y

,pIC50
0,3.928118
1,7.481486
2,7.698970
3,3.602060
4,7.356547
...,...
259,7.619789
260,8.113509
261,7.508638
262,5.000000


## **Combining X and Y variable**

In [21]:
dataset3 = pd.concat([df3_X,df3_Y], axis=1)
dataset3

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,pIC50
0,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,3.928118
1,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.481486
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.698970
3,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,3.602060
4,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.356547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.619789
260,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,8.113509
261,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.508638
262,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,5.000000


In [22]:
dataset3.to_csv('NRF2_Keap1_06_bioactivity_data_3class_pIC50_pubchem_fp.csv', index=False)

# **Let's download the CSV file to your local computer for the Part 3B (Model Building).**